# Optional: an already fine-tuned English sentiment model

Use the PyTorch environment. This checkpoint was fine-tuned on SST-2; it is not an
untrained classification head attached to a base language model. Its scores cannot
be substituted for the IMDb LSTM experiment. The first run downloads model files.

In [1]:
import sys
from importlib.metadata import version
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
assert sys.version_info[:2] == (3, 12), "Select the Python 3.12 project kernel."
assert version('torch').split("+")[0] == '2.14.1', "Select the locked project environment."
print("Python:", sys.version.split()[0])
print('torch' + ":", version('torch'))
print("Run mode:", 'Default classroom training parameters')

Python: 3.12.12
torch: 2.14.1
Run mode: Default classroom training parameters


**Expected output:**

```text
Python: 3.12.12
torch: 2.14.1
Run mode: Default classroom training parameters
```

In [2]:
from pathlib import Path
import os
import sys

# Find the English course root without relying on a specific working directory.
start = Path.cwd().resolve()
COURSE = next((p for p in [start, *start.parents]
               if (p / "COURSE_ROOT").is_file()), None)
if COURSE is None and (start / "english" / "COURSE_ROOT").is_file():
    COURSE = start / "english"
if COURSE is None:
    raise RuntimeError("Start Jupyter inside the project or the english directory.")
sys.path.insert(0, str(COURSE / "common"))
os.environ.setdefault("HF_HOME", str(COURSE / "data" / "huggingface"))
os.environ.setdefault("TORCH_HOME", str(COURSE / "data" / "torch-cache"))

'/Users/will/workspace/deep-learning-lab/english/data/torch-cache'

**Expected output:**

```text
'…/deep-learning-lab/english/data/torch-cache'
```

(Inside the quoted output `…` stands for the repository's absolute path prefix on the machine that ran the notebook; yours will show your own path.)

In [3]:
from transformers import pipeline
import torch
model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
classifier = pipeline("sentiment-analysis", model=model_id, device=DEVICE)
reviews = ["The movie was engaging and beautifully acted.",
           "The story was dull and the ending made no sense."]
for text, result in zip(reviews, classifier(reviews, truncation=True, max_length=512)):
    print(f"{result['label'].lower()} ({result['score']:.3f}): {text}")

/Users/will/env_dl_uv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device set to use mps


positive (1.000): The movie was engaging and beautifully acted.
negative (1.000): The story was dull and the ending made no sense.


**Expected output:**

```text
/Users/will/env_dl_uv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use mps
positive (1.000): The movie was engaging and beautifully acted.
negative (1.000): The story was dull and the ending made no sense.
```

## Discuss

What happens to a review longer than the model limit? How do movie-review and SST-2
examples differ? Why can a high confidence score still be wrong?

[Model card and provenance](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).